# Bài 2 — Bộ nhớ và ngữ cảnh của agent

**Tuần 21** · Bài tập kèm theo: `ex02_bo_nho.py`

Vấn đề mà **mọi** agent đều gặp, và gặp sớm hơn bạn tưởng:

> API là **stateless** (Phase 07). Mỗi bước phải gửi lại toàn bộ lịch sử.
> Bước 15 đắt gấp 15 lần bước 1 — và đến một lúc thì bạn vượt context window
> rồi chết giữa chừng.

Tệ hơn: kết quả công cụ thường **rất dài**. Sau vài bước, phần lớn ngữ cảnh là
dữ liệu model không còn cần nữa.

Không cần API key.

In [ ]:
import sys
from pathlib import Path

GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P09 = GOC / "curriculum" / "09-agents"
sys.path.insert(0, str(P09))

from the_gioi import (CONG_CU_CHI_DOC, CONG_CU_KHONG_HOAN_TAC,
                      CONG_CU_THAY_DOI, TheGioi)

# Windows: console cp1252 lam sap moi lenh print tieng Viet
import sys as _s
for _l in (_s.stdout, _s.stderr):
    if (getattr(_l, "encoding", "") or "").lower().replace("-", "") != "utf8":
        try:
            _l.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

tg = TheGioi()
print(f"{len(tg.don_hang)} đơn hàng · {len(tg.kho)} mặt hàng")
print(f"Công cụ chỉ đọc: {sorted(CONG_CU_CHI_DOC)}")
print(f"Công cụ THAY ĐỔI thế giới: {sorted(CONG_CU_THAY_DOI)}")

In [ ]:
def dang_ky(the_gioi):
    return {
        "tra_cuu_don": the_gioi.tra_cuu_don,
        "tra_cuu_kho": the_gioi.tra_cuu_kho,
        "liet_ke_don": the_gioi.liet_ke_don,
        "cap_nhat_trang_thai": the_gioi.cap_nhat_trang_thai,
        "huy_don": the_gioi.huy_don,
        "dat_hang_bo_sung": the_gioi.dat_hang_bo_sung,
        "gui_email": the_gioi.gui_email,
    }


def thuc_thi(ten, tham_so, so_dang_ky):
    """Khong bao gio nem exception ra ngoai - model se go sai, do la binh thuong."""
    ham = so_dang_ky.get(ten)
    if ham is None:
        return {"ok": False,
                "loi": f"Không có công cụ {ten!r}. Có: {', '.join(sorted(so_dang_ky))}"}
    try:
        return ham(**(tham_so or {}))
    except TypeError as e:
        return {"ok": False, "loi": f"Sai tham so cho {ten}: {e}"}
    except Exception as e:
        return {"ok": False, "loi": f"{type(e).__name__}: {e}"}


def chay_agent(nhiem_vu, goi_model, so_dang_ky, max_vong=6):
    lich_su = [{"vai": "nhiem_vu", "noi_dung": nhiem_vu}]
    so_vong = 0
    for _ in range(max_vong):
        so_vong += 1
        try:
            qd = goi_model(lich_su)
        except Exception:
            return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
                    "ly_do_dung": "loi_model"}
        if not isinstance(qd, dict) or qd.get("loai") not in ("cong_cu", "tra_loi"):
            return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
                    "ly_do_dung": "loi_model"}
        if qd["loai"] == "tra_loi":
            return {"ket_qua": qd.get("noi_dung", ""), "duong_di": lich_su,
                    "so_vong": so_vong, "ly_do_dung": "tra_loi"}
        ten, ts = qd.get("ten", ""), qd.get("tham_so") or {}
        lich_su.append({"vai": "cong_cu", "ten": ten, "tham_so": ts})
        lich_su.append({"vai": "ket_qua", "noi_dung": thuc_thi(ten, ts, so_dang_ky)})
    return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
            "ly_do_dung": "het_vong"}


def in_duong_di(kq):
    n = 0
    ket_qua = [b["noi_dung"] for b in kq["duong_di"] if b["vai"] == "ket_qua"]
    for b in kq["duong_di"]:
        if b["vai"] != "cong_cu":
            continue
        ts = ", ".join(f"{k}={v!r}" for k, v in b["tham_so"].items())
        r = ket_qua[n] if n < len(ket_qua) else {}
        dau = "ok" if r.get("ok") else f'LOI: {r.get("loi", "?")}'
        n += 1
        print(f"{n}. {b['ten']}({ts}) -> {dau}")
    print(f"=> {kq['ket_qua'] or '(không có câu trả lời)'}   [{kq['ly_do_dung']}]")

## 1. Chi phí tăng theo bình phương

Hãy đo thật, đừng tin cảm giác.

In [ ]:
def dem_token_uoc(lich_su):
    """Ước lượng: tổng ký tự / 3. Con số CHÍNH XÁC phải từ count_tokens()."""
    return sum(len(str(b)) for b in lich_su) // 3


def tao_lich_su(so_buoc, dai_ket_qua=400):
    ls = [{"vai": "nhiem_vu", "noi_dung": "Rà soát toàn bộ kho hàng"}]
    for i in range(so_buoc):
        ls.append({"vai": "cong_cu", "ten": "tra_cuu_kho",
                   "tham_so": {"ma_hang": f"VX-{i:02d}"}})
        ls.append({"vai": "ket_qua",
                   "noi_dung": {"ok": True, "ket_qua": "x" * dai_ket_qua}})
    return ls


GIA_VAO = 1.00 / 1_000_000        # Haiku 4.5

print(f'{"bước":>5}{"token gửi ở bước này":>24}{"TỔNG token đã gửi":>20}{"chi phí":>11}')
cong_don = 0
for buoc in range(1, 16):
    tok = dem_token_uoc(tao_lich_su(buoc))
    cong_don += tok
    if buoc in (1, 3, 5, 10, 15):
        print(f"{buoc:>5}{tok:>24,}{cong_don:>20,}{cong_don * GIA_VAO:>11.4f}")

Cột giữa là điểm mấu chốt: một nhiệm vụ 15 bước **không** tốn gấp 15 lần một
bước — nó tốn gấp khoảng **100 lần**, vì mỗi bước gửi lại tất cả những gì đã qua.

> 📌 Đây là lý do agent đắt hơn nhiều so với dự đoán đầu tiên của mọi người. Và
> nó cũng giải thích vì sao `max_vong` vừa là chốt an toàn, vừa là chốt chi phí.

## 2. Ba chiến lược, từ ít mất mát đến nhiều mất mát

| Chiến lược | Giữ được gì | Mất gì | Tốn thêm |
|---|---|---|---|
| **Rút gọn** | Cấu trúc + ý nghĩa | Chi tiết trong kết quả | Không |
| **Tóm tắt** | Ý nghĩa | Chi tiết | **Một lần gọi API** |
| **Cắt bỏ** | Các bước gần nhất | Toàn bộ bước cũ | Không |

**Luôn dùng biện pháp ít mất mát nhất trước.** Người mới hay làm ngược — cắt bỏ
trước vì dễ code nhất, rồi ngạc nhiên khi agent quên mất việc mình vừa làm.

In [ ]:
DAU_RUT_GON = "... (đã rút gọn)"


def rut_gon_ket_qua(lich_su, max_ky_tu=200):
    moi = []
    for b in lich_su:
        if b.get("vai") != "ket_qua" or not isinstance(b.get("noi_dung"), dict):
            moi.append(dict(b))
            continue
        nd = {k: (v[:max_ky_tu] + DAU_RUT_GON
                  if isinstance(v, str) and len(v) > max_ky_tu else v)
              for k, v in b["noi_dung"].items()}
        moi.append({**b, "noi_dung": nd})
    return moi


def _tach_cap(lich_su):
    dau = [b for b in lich_su if b.get("vai") in ("nhiem_vu", "tom_tat")]
    con = [b for b in lich_su if b.get("vai") in ("cong_cu", "ket_qua")]
    return dau, [con[i:i + 2] for i in range(0, len(con), 2)]


def cat_lich_su(lich_su, giu_cap=3):
    dau, cap = _tach_cap(lich_su)
    giu = cap[-giu_cap:] if giu_cap else []
    return dau + [b for c in giu for b in c]


ls = tao_lich_su(10)
print(f'{"nguyên bản":<22}{dem_token_uoc(ls):>8,} token')
print(f'{"rút gọn (200)":<22}{dem_token_uoc(rut_gon_ket_qua(ls, 200)):>8,} token')
print(f'{"cắt (giữ 3 cặp)":<22}{dem_token_uoc(cat_lich_su(ls, 3)):>8,} token')
ca_hai = cat_lich_su(rut_gon_ket_qua(ls, 200), 3)
print(f'{"rút gọn + cắt":<22}{dem_token_uoc(ca_hai):>8,} token')

## 3. Cái giá của việc cắt bỏ — một demo đáng nhớ

Cắt bỏ rẻ và hiệu quả. Nhưng nó có một cái giá **rất cụ thể**.

In [ ]:
tg = TheGioi()
sdk = dang_ky(tg)

def model_quen(lich_su):
    """Model chỉ nhìn thấy lịch sử được đưa cho nó.

    Nếu bước 'đã huỷ DH1002' bị cắt mất, nó không có cách nào biết việc đó
    đã làm rồi.
    """
    da_huy = any(b.get("ten") == "huy_don" for b in lich_su if b["vai"] == "cong_cu")
    if da_huy:
        return {"loai": "tra_loi", "noi_dung": "Đã huỷ đơn DH1002 rồi."}
    return {"loai": "cong_cu", "ten": "huy_don", "tham_so": {"ma_don": "DH1002"}}


# Lịch sử ĐẦY ĐỦ: model thấy được việc mình vừa làm
ls_du = [{"vai": "nhiem_vu", "noi_dung": "Huỷ đơn DH1002"},
         {"vai": "cong_cu", "ten": "huy_don", "tham_so": {"ma_don": "DH1002"}},
         {"vai": "ket_qua", "noi_dung": {"ok": True, "ket_qua": "Đã huỷ DH1002"}}]
print("Lịch sử đầy đủ  ->", model_quen(ls_du))

# Sau khi CẮT: bước huỷ biến mất
ls_cat = cat_lich_su(ls_du, giu_cap=0)
print("Sau khi cắt     ->", model_quen(ls_cat))

> ⚠️ **Agent vừa quyết định huỷ đơn LẦN THỨ HAI.**
>
> Với `huy_don`, thế giới của chúng ta **idempotent** — gọi lần hai chỉ báo lỗi
> "đã huỷ từ trước", không hại thêm. Nhưng với `gui_email` thì khách hàng nhận
> hai email. Với `dat_hang_bo_sung` thì bạn mua hàng hai lần.
>
> **Đây là lý do hai thiết kế sau không phải là tuỳ chọn:**
>
> | Nguyên tắc | Vì sao |
> |---|---|
> | **Công cụ idempotent** | Gọi hai lần không hại thêm lần nào |
> | **Nhật ký riêng, ngoài lịch sử gửi model** | Bạn vẫn biết đã làm gì, kể cả sau khi cắt |
>
> Thế giới của chúng ta có `tg.nhat_ky` chính vì lý do này — nó **không bị cắt**
> cùng với ngữ cảnh gửi cho model.

In [ ]:
# Agent đã THẬT SỰ huỷ đơn (bước đầu tiên ở trên), giờ cắt lịch sử đi
tg.huy_don("DH1002")

print("Lịch sử gửi cho model còn:", len(ls_cat), "phần tử  <- bước huỷ BIẾN MẤT")
print("Nhật ký thế giới còn     :", len(tg.nhat_ky), "bản ghi  <- vẫn còn nguyên")
print()
print("Kiểm tra trước khi hành động - luôn rẻ hơn là làm nhầm:")
da_tung_huy = any(g.cong_cu == "huy_don" and g.thanh_cong for g in tg.nhat_ky)
print(f"   Đã từng huỷ đơn trong phiên này? {da_tung_huy}")

## 4. Tóm tắt — và đường lui khi nó hỏng

In [ ]:
def tom_tat_cu(lich_su, giu_cap, ham_tom_tat):
    dau, cap = _tach_cap(lich_su)
    if giu_cap >= len(cap):
        return list(lich_su)                     # không có gì để tóm tắt
    cu = [b for c in cap[:len(cap) - giu_cap] for b in c]
    try:
        noi_dung = ham_tom_tat(cu)
    except Exception:
        # Tóm tắt là MỘT LẦN GỌI API nữa - nó có thể rate limit, timeout,
        # hết quota. Đừng biến một tối ưu chi phí thành một điểm hỏng mới.
        return cat_lich_su(lich_su, giu_cap)
    return dau + [{"vai": "tom_tat", "noi_dung": noi_dung}] + \
           [b for c in cap[len(cap) - giu_cap:] for b in c]


ls = tao_lich_su(8)
tot = tom_tat_cu(ls, 2, lambda cu: f"Đã tra cứu {len(cu)//2} mặt hàng, đều đủ tồn.")
hong = tom_tat_cu(ls, 2, lambda cu: (_ for _ in ()).throw(RuntimeError("rate limit")))

print(f'{"nguyên bản":<28}{dem_token_uoc(ls):>7,} token   {len(ls)} phần tử')
print(f'{"tóm tắt được":<28}{dem_token_uoc(tot):>7,} token   {len(tot)} phần tử')
print(f'{"tóm tắt hỏng -> lùi về cắt":<28}{dem_token_uoc(hong):>7,} token   {len(hong)} phần tử')
print()
print("Khối tóm tắt:", [b for b in tot if b["vai"] == "tom_tat"])

## 5. Ghép lại: nén cho vừa ngân sách

In [ ]:
def nen_de_vua(lich_su, ngan_sach, ham_tom_tat=None):
    da_dung, ht = [], list(lich_su)
    if dem_token_uoc(ht) <= ngan_sach:
        return ht, da_dung
    ht = rut_gon_ket_qua(ht, 200);  da_dung.append("rut_gon")
    if dem_token_uoc(ht) <= ngan_sach:
        return ht, da_dung
    if ham_tom_tat is not None:
        ht = tom_tat_cu(ht, 2, ham_tom_tat);  da_dung.append("tom_tat")
        if dem_token_uoc(ht) <= ngan_sach:
            return ht, da_dung
    ht = cat_lich_su(ht, 2);  da_dung.append("cat")
    return ht, da_dung


ls = tao_lich_su(12, 600)
print(f"Trước khi nén: {dem_token_uoc(ls):,} token")
for ngan_sach in (20000, 3000, 800, 200):
    moi, chien_luoc = nen_de_vua(ls, ngan_sach, lambda cu: "Đã rà 10 mặt hàng.")
    vua = "✅" if dem_token_uoc(moi) <= ngan_sach else "❌ vẫn không vừa"
    print(f"  ngân sách {ngan_sach:>6,} -> {dem_token_uoc(moi):>6,} token  "
          f"{vua:<16} dùng: {chien_luoc}")

Dòng cuối cùng đáng chú ý: **có ngân sách nhỏ đến mức không chiến lược nào
đáp ứng nổi.** Lúc đó lựa chọn đúng là để model tự báo *"tôi không làm được"* chứ
không phải làm sập chương trình — nên `nen_de_vua` vẫn trả về kết quả kèm danh
sách những gì đã thử.

> 📌 Trong Claude API thật, bạn còn hai công cụ nữa cho việc này, đều là beta:
> **context editing** (`clear_tool_uses`) tự xoá kết quả công cụ cũ, và
> **compaction** tự tóm tắt phần đầu hội thoại. Cả hai làm đúng việc bạn vừa tự
> viết — nhưng **tự viết một lần** rồi mới dùng bản có sẵn thì bạn biết nó đang
> làm gì và hỏng thế nào.

## 6. Phát hiện agent bị kẹt

In [ ]:
def phat_hien_lap(lich_su, nguong=2):
    dem = {}
    for b in lich_su:
        if b.get("vai") != "cong_cu":
            continue
        # dict không băm được -> khoá phải SẮP XẾP, nếu không {"a":1,"b":2} và
        # {"b":2,"a":1} thành hai khoá khác nhau và bạn bỏ sót đúng cái lặp.
        khoa = f'{b["ten"]}|{sorted((b.get("tham_so") or {}).items())}'
        dem.setdefault(khoa, {"ten": b["ten"], "tham_so": b.get("tham_so") or {},
                              "so_lan": 0})
        dem[khoa]["so_lan"] += 1
    return sorted([v for v in dem.values() if v["so_lan"] >= nguong],
                  key=lambda v: -v["so_lan"])


tg = TheGioi()
kq = chay_agent("Huỷ đơn DH1003",
                lambda ls: {"loai": "cong_cu", "ten": "huy_don",
                            "tham_so": {"ma_don": "DH1003"}},
                dang_ky(tg), max_vong=6)

print(f'Agent dừng vì: {kq["ly_do_dung"]} sau {kq["so_vong"]} vòng')
for lap in phat_hien_lap(kq["duong_di"], nguong=2):
    print(f'   ⚠️  {lap["ten"]}({lap["tham_so"]}) gọi {lap["so_lan"]} lần y hệt nhau')

Phát hiện sớm cho bạn ba lựa chọn, thay vì ngồi đợi hết `max_vong`:

| Cách xử lý | Khi nào |
|---|---|
| Chèn gợi ý vào lịch sử *("bạn đã thử cách này rồi")* | Lặp 2–3 lần |
| Đổi chiến lược / đổi model | Lặp nhiều lần ở cùng một chỗ |
| **Dừng sớm và báo người** | Việc quan trọng, hoặc lặp ở hành động có hậu quả |

> 📌 Ba lần lặp phát hiện ở vòng 3 rẻ hơn sáu lần phát hiện ở vòng 6 — và với
> agent chạy hàng nghìn phiên mỗi ngày, chênh lệch đó là tiền thật.

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Chi phí bình phương** | 15 bước tốn ~100 lần một bước, không phải 15 lần |
| **Thứ tự chiến lược** | Rút gọn → tóm tắt → cắt. Ít mất mát trước |
| **Cắt có giá** | Agent quên việc đã làm và làm lại — email gửi hai lần |
| **Idempotent + nhật ký ngoài** | Hai thiết kế bắt buộc, không phải tuỳ chọn |
| **Tóm tắt có thể hỏng** | Nó là một lần gọi API — phải có đường lui |
| **Phát hiện lặp** | Rẻ, dễ, và cứu bạn khỏi những hoá đơn khó giải thích |

**Bài tập:** `curriculum/09-agents/exercises/ex02_bo_nho.py`

```powershell
pytest tests/phase09/test_ex02.py -v
```

➡️ Bài tiếp: **[03 — Guardrails và hành động nguy hiểm](03_guardrails.ipynb)**